# MP1 analysis — Assignment 5

**Notebook file:** `MP1 analysis A5.ipynb` (this file).

**This notebook has 13 cells.** Scroll down: after each `##` heading you will see a **code** cell (Python). Run cells from top to bottom.

**CSV:** `380K_US_Restaurants.csv` in the same folder as this notebook (or `380K_UW_Restaurants.csv`). The first code cell tries a few paths so it can still find the file if your working directory is the `HCDE 530` folder.

In [ ]:
import pandas as pd
from pathlib import Path

# Load every row so counts and percentages reflect the full export (not a sample).
# Resolve CSV whether your notebook cwd is this folder or the HCDE 530 workspace root.
_candidates = [
    Path("380K_US_Restaurants.csv"),
    Path("380K_UW_Restaurants.csv"),
    Path("MP1 Restaurants") / "380K_US_Restaurants.csv",
    Path("MP1 Restaurants") / "380K_UW_Restaurants.csv",
]
_csv = next((p for p in _candidates if p.exists()), None)
if _csv is None:
    raise FileNotFoundError("Could not find 380K_US_Restaurants.csv or 380K_UW_Restaurants.csv")
df = pd.read_csv(_csv, low_memory=False)
# Ratings should be numeric for averages; coerce bad strings to NaN so they do not break groupby.
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
print(f"Loaded {len(df):,} rows from {_csv.resolve()}")

## First look — what does each row represent?

Each row is one Google Maps listing: name, primary **`Category`**, **`Rating`**, address, **`Time_Zone`**, etc.

In [ ]:
# Question for the data: what do real rows look like — which fields are filled, and do categories/ratings look plausible?
# Answer means: you can sanity-check the scrape (e.g. one restaurant per row, mixed text in Images) before trusting aggregates.
df.head()

In [ ]:
# Question for the data: how big is the table, what are the column types, and are any columns already partly empty at load time?
# Answer means: row count matches expectations; dtypes tell you what can be averaged vs only counted.
df.info()

## Missing values — which fields are incomplete?

Use this when interpreting averages or website-based questions: lots of missing values in a column mean summaries only describe rows where that field is present.

In [ ]:
# Question for the data: for each column, how many cells are missing — and does that limit how we read ratings, websites, or time zones?
# Answer means: high missing counts flag columns where "average rating" or "has a website" only describes places that reported that field.
df.isnull().sum().sort_values(ascending=False)

## Question 1 — How many category labels exist, and how common is each?

We use Google's primary **`Category`** field (one label per row). **Count** = how many listings use that label; **percentage** = share of **all rows** in this file.

In [ ]:
# Question for the data: how many distinct primary category strings appear, and what share of the dataset does each category represent?
# Answer means: a high count of unique categories says the map data is very fragmented; a few huge percentages say the file is dominated by a few restaurant types.
n_cat = df["Category"].nunique(dropna=True)
print(f"Distinct Category values (non-null): {n_cat:,}")

counts = df["Category"].value_counts(dropna=False)
pct = (counts / len(df) * 100).round(2)
category_distribution = pd.DataFrame({"count": counts, "pct_of_all_rows": pct})
# Built from value_counts: one row per Category label with count and % of all rows.
category_distribution

## Question 2 — Among Asian-related cuisines, which primary labels are most common?

We keep rows whose **`Category`** text matches **Asian / Chinese / Vietnamese / Thai / Indian** (case-insensitive). Then we rank those **exact** category strings by listing count and by **percent of this subset** (not of the whole file).

In [ ]:
# Question for the data: within Asian-related labels, which official category name appears most often — e.g. more "Chinese restaurant" than "Thai restaurant"?
# Answer means: "most popular" here is row count in the data, not revenue or quality; takeaway vs dine-in can appear as different labels.
asian_pattern = r"(?:asian|chinese|vietnamese|thai|indian)"
asian_related = df[df["Category"].str.contains(asian_pattern, case=False, na=False, regex=True)]
print(f"Rows matching Asian / Chinese / Vietnamese / Thai / Indian in Category: {len(asian_related):,}")

# value_counts on the filtered slice: distribution within Asian-related rows only.
_vc = asian_related["Category"].value_counts()
_pct_slice = (_vc / len(asian_related) * 100).round(2)
asian_cuisine_distribution = pd.DataFrame({"count": _vc, "pct_of_asian_slice": _pct_slice})
asian_cuisine_distribution

## Question 3 — By time zone, how many restaurants and what is the average rating?

**`Time_Zone`** is the IANA time zone (mostly US, some international). **`n_listings`** counts rows per zone; **`avg_rating`** is the mean of **`Rating`** in that zone (pandas skips NaN ratings in the mean). Same idea as `df.groupby("Time_Zone")["Rating"].mean()` plus a row count.

In [ ]:
# Question for the data: when we group all rows by Time_Zone, where are most listings, and do average ratings differ by zone in this export?
# Answer means: zones with huge n drive overall patterns; comparing avg_rating suggests where stars look higher/lower (watch missing ratings and mixed geography per zone).
(
    df.groupby("Time_Zone", dropna=False)
    .agg(n_listings=("Title", "size"), avg_rating=("Rating", "mean"))
    .sort_values("n_listings", ascending=False)
    .round({"avg_rating": 3})
)